#  ViT5 Fine-tuning — Chuẩn Hóa Tin Đăng Bất Động Sản

**Thực hiện:** Đinh Thị Mai Anh — ViT5 Specialist

**Môn học:** NLP501 — FSB | Nhóm 2

**Mô hình:** `VietAI/vit5-base` (220M params) — Seq2Seq Transformer tiếng Việt

**Môi trường:** Google Colab + GPU NVIDIA T4

---

##  Mục tiêu
Fine-tune mô hình ViT5 để tự động **rút gọn & chuẩn hóa** tin đăng BĐS thô:
- **Ẩn** thông tin nhạy cảm: số nhà, SĐT, mã hoa hồng môi giới
- **Giữ nguyên** 9 specs kỹ thuật: đường, quận, giá, diện tích, số tầng...
- **Viết lại** theo văn phong chuyên nghiệp, cuốn hút

##  Dataset
| Tập | Số mẫu | Ghi chú |
|---|---|---|
| Train | 9 mẫu | Có Ground Truth |
| Val | 2 mẫu | Có Ground Truth |
| Test | 9 mẫu | Không có GT — đánh giá định tính |

##  Cách sử dụng notebook
Chạy **tuần tự từ trên xuống**, không bỏ cell nào.

##  Bước 1: Cài Đặt Thư Viện

Cài tất cả thư viện cần thiết cho dự án.
Chỉ cần chạy **1 lần** khi bắt đầu session Colab mới.

| Thư viện | Công dụng |
|---|---|
| `transformers` | Load và fine-tune mô hình ViT5 (HuggingFace) |
| `datasets` | Quản lý dữ liệu theo chuẩn HuggingFace Dataset |
| `sentencepiece` | Tokenizer tiếng Việt cho T5/ViT5 |
| `rouge_score` | Tính điểm ROUGE-L đánh giá chất lượng |
| `evaluate` | Framework đánh giá chuẩn HuggingFace |
| `accelerate` | Tối ưu hóa training trên GPU |

In [1]:
# Cài đặt tất cả thư viện cần thiết
# Cập nhật transformers lên version mới hơn để tránh lỗi EncoderDecoderCache
!pip install -q transformers==4.44.2 datasets sentencepiece peft
!pip install -q rouge_score evaluate accelerate

print(' Cài đặt thư viện hoàn tất!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 126.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00
 Cài đặt thư viện hoàn tất!


##  Bước 2: Import Thư Viện & Kiểm Tra GPU

Import toàn bộ thư viện và kiểm tra xem Colab có GPU T4 không.

>  **Quan trọng:** Nếu thấy thông báo *'KHÔNG CÓ GPU'*, vào **Runtime → Change runtime type → GPU (T4)** trước khi tiếp tục.

In [2]:
import os, json, time, random, warnings, re
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Tránh lỗi display trên Colab
import matplotlib.pyplot as plt

import torch
from transformers import (
    AutoTokenizer,           # Tokenizer tự động theo model
    AutoModelForSeq2SeqLM,  # Load mô hình Seq2Seq (ViT5)
    Seq2SeqTrainer,          # Trainer tối ưu cho bài toán Seq2Seq
    Seq2SeqTrainingArguments,# Cấu hình training
    DataCollatorForSeq2Seq, # Padding động (tiết kiệm RAM)
    EarlyStoppingCallback,  # Dừng sớm khi val_loss không cải thiện
    GenerationConfig,        # Cấu hình Beam Search khi inference
)
from datasets import Dataset   # Định dạng Dataset của HuggingFace
import evaluate                # Thư viện tính ROUGE, BERTScore...
from google.colab import files # Upload/download file trong Colab

warnings.filterwarnings('ignore')  # Tắt cảnh báo không quan trọng

# ── Kiểm tra GPU ──────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f' GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print(' KHÔNG CÓ GPU! Vào Runtime → Change runtime type → GPU (T4)')

# ── Cố định seed để kết quả tái hiện được ────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'\n Import hoàn tất | Device: {device}')

 GPU: Tesla T4
   VRAM: 15.6 GB

 Import hoàn tất | Device: cuda


##  Bước 3: Cấu Hình Hyperparameters

Tất cả tham số quan trọng đều đặt ở đây. **Chỉ cần chỉnh tại cell này.**

### Lý do chọn từng tham số (dataset chỉ có 9 mẫu train):
- **Learning Rate = 1e-4** (nhỏ): tránh nhảy quá lớn khi data ít
- **Epochs = 80** (nhiều): data ít → cần nhiều lượt để học
- **Batch Size = 2** (nhỏ): phù hợp với 9 mẫu, tránh batch lớn hơn tập data
- **EarlyStopping = 15** (kiên nhẫn): val chỉ có 2 mẫu → loss dao động nhiều
- **Weight Decay = 0.05** (cao): regularization mạnh để chống overfit
- **Beam Size = 4**: tìm câu tốt nhất bằng cách thử 4 hướng song song

In [3]:
# ── Tên mô hình gốc trên HuggingFace ─────────────────────────
MODEL_NAME = 'VietAI/vit5-base'  # ViT5-base: 220M params, chuyên tiếng Việt

# ── Hyperparameters Training ──────────────────────────────────
EPOCHS        = 80     # Số epoch tối đa (EarlyStopping sẽ dừng sớm hơn)
LEARNING_RATE = 1e-4   # Tốc độ học — nhỏ để tránh overshoot với data ít
BATCH_SIZE    = 2      # Số mẫu mỗi batch — nhỏ phù hợp 9 mẫu
GRAD_ACCUM    = 4      # Tích lũy gradient — effective batch = 2x4 = 8
WARMUP_STEPS  = 10     # Bước khởi động — tăng LR dần từ 0
WEIGHT_DECAY  = 0.05   # L2 regularization — chống overfit
PATIENCE      = 15     # EarlyStopping: dừng nếu val_loss không giảm 15 epoch liên tiếp

# ── Cấu hình Tokenizer ────────────────────────────────────────
MAX_INPUT_LEN  = 384   # Độ dài tối đa input (mô tả thô thường rất dài)
MAX_TARGET_LEN = 256   # Độ dài tối đa output (mô tả chuẩn BĐS)

# ── Cấu hình Beam Search (khi inference) ─────────────────────
BEAM_SIZE       = 4    # Số beam — thử 1-5 ở Cell 15 để chọn tối ưu
LENGTH_PENALTY  = 1.2  # > 1 → ưu tiên câu dài hơn (mô tả đầy đủ hơn)
NO_REPEAT_NGRAM = 3    # Cấm lặp lại chuỗi 3 từ liên tiếp

# ── Task Prefix ────────────────────────────────────────────────
# T5/ViT5 hoạt động theo cơ chế text-to-text → cần prefix để chỉ định task
TASK_PREFIX = 'chuan hoa tin bat dong san: '

# ── Thư mục lưu checkpoint ────────────────────────────────────
OUTPUT_DIR = './vit5_checkpoint'   # Lưu tất cả checkpoint
FINAL_DIR  = './vit5_best'         # Chỉ lưu checkpoint tốt nhất

# ── Augmentation ──────────────────────────────────────────────
AUGMENT = True  # True: bật data augmentation (nên để True với data ít)

print(' Hyperparameters đã cấu hình:')
print(f'   Model:          {MODEL_NAME}')
print(f'   Epochs (max):   {EPOCHS} | EarlyStopping patience={PATIENCE}')
print(f'   Learning Rate:  {LEARNING_RATE}')
print(f'   Batch (hiệu dụng): {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}')
print(f'   Input/Output:   {MAX_INPUT_LEN} / {MAX_TARGET_LEN} tokens')
print(f'   Beam Size:      {BEAM_SIZE}')
print(f'   Augmentation:   {AUGMENT}')

 Hyperparameters đã cấu hình:
   Model:          VietAI/vit5-base
   Epochs (max):   80 | EarlyStopping patience=15
   Learning Rate:  0.0001
   Batch (hiệu dụng): 2 x 4 = 8
   Input/Output:   384 / 256 tokens
   Beam Size:      4
   Augmentation:   True


##  Bước 4: Upload Dữ Liệu

Upload 3 file JSON đã tạo từ `prepare_dataset.py`:

| File | Mẫu | Nội dung |
|---|---|---|
| `train.json` | 9 mẫu | Có Ground Truth → huấn luyện ViT5 |
| `val.json` | 2 mẫu | Có Ground Truth → theo dõi val_loss |
| `test.json` | 9 mẫu | Không có GT → inference ra kết quả cuối |

> 📌 Nhấn nút **Choose Files** xuất hiện bên dưới và chọn cả 3 file cùng lúc.

In [4]:
# Upload 3 file dữ liệu từ máy tính lên Colab
print(' Chọn 3 file: train.json, val.json, test.json...')
uploaded = files.upload()

# In ra tên các file đã upload
for fname in uploaded.keys():
    print(f'   Đã upload: {fname}')

# Đường dẫn cố định (file được upload vào /content/)
TRAIN_PATH = 'train.json'
VAL_PATH   = 'val.json'
TEST_PATH  = 'test.json'

print('\n Sẵn sàng load dữ liệu!')

 Chọn 3 file: train.json, val.json, test.json...


Saving test.json to test (1).json
Saving train.json to train (1).json
Saving val.json to val (1).json
   Đã upload: test (1).json
   Đã upload: train (1).json
   Đã upload: val (1).json

 Sẵn sàng load dữ liệu!


##  Bước 5: Load & Kiểm Tra Dữ Liệu

Đọc file JSON, chuẩn hóa tên cột và in mẫu để kiểm tra dữ liệu đúng định dạng.

**Cấu trúc 1 mẫu dữ liệu:**
```json
{
  "id": "SYS-MP759ZIK-MM",
  "raw_input_cleaned": "Nguyễn Thiện Thuật 30.7 2 3.1 11...",  // Mô tả thô
  "clean_description": "Nguyễn Thiện Thuật Quận 3 - 31m2...",  // Ground Truth
  "extracted_specs": { "duong": "...", "gia_ty": 8.2, ... }
}
```

In [5]:
def load_json(path):
    """Đọc file JSON với encoding UTF-8."""
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def parse_sample(item):
    """
    Parse 1 mẫu từ JSON.
    Hàm tự động nhận diện nhiều tên cột khác nhau
    (linh hoạt với nhiều định dạng file ).
    """
    # Lấy mô tả thô (input cho mô hình)
    raw = (
        item.get('raw_input_cleaned') or
        item.get('mo_ta_tho') or
        item.get('input') or ''
    ).strip()

    # Lấy mô tả chuẩn (target / ground truth)
    desc = (
        item.get('clean_description') or
        item.get('mo_ta_chuan') or
        item.get('target') or ''
    ).strip()

    # Lấy tiêu đề chuẩn (nếu có)
    title = (
        item.get('clean_title') or
        item.get('tieu_de_chuan') or ''
    ).strip()

    # Kết hợp title + description làm target text cho model
    # ViT5 sẽ học sinh ra cả 2 trong 1 output
    if title and desc:
        target = f'Tieu de: {title}\nMo ta: {desc}'
    elif title:
        target = title
    else:
        target = desc

    return {
        'id':          item.get('id', ''),
        'input_text':  raw,
        'target_text': target,
        'raw_item':    item,   # Giữ nguyên để lấy specs sau
    }

# Load 3 tập dữ liệu
raw_train = [parse_sample(x) for x in load_json(TRAIN_PATH)]
raw_val   = [parse_sample(x) for x in load_json(VAL_PATH)]
raw_test  = [parse_sample(x) for x in load_json(TEST_PATH)]

print(f'📊 Thống kê dữ liệu:')
print(f'   Train: {len(raw_train)} mẫu (có Ground Truth)')
print(f'   Val:   {len(raw_val)} mẫu (có Ground Truth)')
print(f'   Test:  {len(raw_test)} mẫu (không có GT)')

# In mẫu đầu tiên để kiểm tra
print('\n' + '='*60)
print('📌 MẪU TRAIN [0] — Kiểm tra dữ liệu:')
print('='*60)
s = raw_train[0]
print(f'ID:     {s["id"]}')
print(f'INPUT:  {s["input_text"][:150]}...')
print(f'TARGET: {s["target_text"][:150]}...')

# Thống kê độ dài
in_lens  = [len(s['input_text'].split())  for s in raw_train]
out_lens = [len(s['target_text'].split()) for s in raw_train]
print(f'\n📏 Độ dài trung bình (train):')
print(f'   Input:  {np.mean(in_lens):.0f} từ (max={max(in_lens)})')
print(f'   Target: {np.mean(out_lens):.0f} từ (max={max(out_lens)})')

📊 Thống kê dữ liệu:
   Train: 9 mẫu (có Ground Truth)
   Val:   2 mẫu (có Ground Truth)
   Test:  26 mẫu (không có GT)

📌 MẪU TRAIN [0] — Kiểm tra dữ liệu:
ID:     SYS-MP759ZIK-MM
INPUT:  Nguyễn Thiện Thuật 30.7 2 3.1 11 8.2 tỷ Bàn Cờ Quận 3 5-10 tỷ HĐ Nguyễn Thị Kim Thoa_ĐC Linh Phát  H3GB nguồn Đầu chủ Nguyễn Thị Kim Thoa_ĐC - Linh Ph...
TARGET: Nguyễn Thiện Thuật Quận 3 - 31m2 - Hẻm xe tải thông kinh doanh - 8.2 Tỷ | Ở ngay hoặc Kinh doanh

KHU VỰC KINH DOANH SẦM UẤT NGUYỄN THIỆN THUẬT QUẬN 3...

📏 Độ dài trung bình (train):
   Input:  189 từ (max=258)
   Target: 233 từ (max=387)


##  Bước 6: Load ViT5 Model & Tokenizer

Download mô hình `VietAI/vit5-base` từ HuggingFace (~900MB).

**Lần đầu chạy:** Mất khoảng **2-3 phút** để download.

**ViT5 là gì?**
- Viết tắt của **Vietnamese T5** — phiên bản T5 được pre-train trên dữ liệu tiếng Việt khổng lồ
- Kiến trúc **Encoder-Decoder** (Seq2Seq): Encoder đọc hiểu văn bản thô → Decoder sinh ra văn bản chuẩn
- Pre-trained bởi nhóm VietAI, tối ưu cho các tác vụ NLP tiếng Việt

In [6]:
print(f'⏳ Đang download mô hình "{MODEL_NAME}"...')
print('   (Lần đầu mất ~2-3 phút, các lần sau cache lại nhanh hơn)\n')

# Load tokenizer: chuyển văn bản → token IDs mà model hiểu được
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load model: kiến trúc Seq2Seq (Encoder-Decoder)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# Chuyển model lên GPU để training nhanh hơn
model = model.to(device)

# Thống kê số tham số
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f' Load model thành công!')
print(f'   Tổng params:      {total_params/1e6:.1f}M')
print(f'   Trainable params: {trainable_params/1e6:.1f}M')
print(f'   Model device:     {next(model.parameters()).device}')

# Thử tokenize 1 câu để kiểm tra tokenizer hoạt động
test_tokens = tokenizer('Nhà đẹp Quận 3', return_tensors='pt')
print(f'\n Test tokenizer: "Nhà đẹp Quận 3" → {test_tokens["input_ids"].shape[1]} tokens')

⏳ Đang download mô hình "VietAI/vit5-base"...
   (Lần đầu mất ~2-3 phút, các lần sau cache lại nhanh hơn)



tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

 Load model thành công!
   Tổng params:      226.0M
   Trainable params: 226.0M
   Model device:     cuda:0

 Test tokenizer: "Nhà đẹp Quận 3" → 5 tokens


## Bước 7: Tokenize Dữ Liệu

Chuyển văn bản thô thành dãy số (token IDs) mà mô hình ViT5 hiểu được.

**Quá trình tokenize:**
```
"Nhà 5 tầng Quận 10"  →  [3, 1234, 56, 789, 10, 1]  (token IDs)
```

**Task Prefix:** Thêm `'chuan hoa tin bat dong san: '` vào đầu mỗi input.
T5/ViT5 hoạt động theo cơ chế **text-to-text** — cần prefix để mô hình biết đang làm task gì.

In [7]:
def tokenize_function(examples):
    """
    Tokenize batch input và target.
    Hàm này được gọi tự động bởi Dataset.map().
    """
    # Thêm task prefix vào đầu mỗi input
    inputs = [TASK_PREFIX + text for text in examples['input_text']]

    # Tokenize input (mô tả thô)
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        truncation=True,    # Cắt bớt nếu quá dài
        padding=False,      # Padding sẽ do DataCollator xử lý động
    )

    # Tokenize target (mô tả chuẩn = ground truth)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples['target_text'],
            max_length=MAX_TARGET_LEN,
            truncation=True,
            padding=False,
        )

    # Gán labels (target token IDs) vào model_inputs
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

def to_hf_dataset(raw_list):
    """Chuyển list dict Python → HuggingFace Dataset rồi tokenize."""
    ds = Dataset.from_dict({
        'input_text':  [s['input_text']  for s in raw_list],
        'target_text': [s['target_text'] for s in raw_list],
    })
    return ds.map(
        tokenize_function,
        batched=True,                              # Xử lý theo batch (nhanh hơn)
        remove_columns=['input_text', 'target_text'],  # Xóa cột text sau khi tokenize
    )

# Tokenize 3 tập dữ liệu
train_ds = to_hf_dataset(raw_train)
val_ds   = to_hf_dataset(raw_val)

print('✅ Tokenize hoàn tất!')
print(f'   Train: {len(train_ds)} samples')
print(f'   Val:   {len(val_ds)} samples')

# Kiểm tra 1 mẫu tokenized
sample = train_ds[0]
print(f'\n📏 Mẫu train[0] sau tokenize:')
print(f'   input_ids:  {len(sample["input_ids"])} tokens')
print(f'   labels:     {len(sample["labels"])} tokens')
print(f'   Decode thử: "{tokenizer.decode(sample["input_ids"][:20])}..."')

Map:   0%|          | 0/9 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

✅ Tokenize hoàn tất!
   Train: 9 samples
   Val:   2 samples

📏 Mẫu train[0] sau tokenize:
   input_ids:  145 tokens
   labels:     256 tokens
   Decode thử: "chuan hoa tin bat dong san: Nguyễn Thiện Thuật 30.7 2 3.1 1..."


##  Bước 8: Data Augmentation (Tăng Cường Dữ Liệu)

Với chỉ **9 mẫu train**, mô hình ViT5  220M params rất dễ **overfit** (học thuộc lòng training data).

**Giải pháp: Synonym Swapping** — Thay thế từ đồng nghĩa trong ngành BĐS:

| Từ gốc | Từ thay thế |
|---|---|
| xe hơi | ô tô |
| tầng | lầu |
| WC | nhà vệ sinh |
| thương lượng | TL |
| phòng ngủ | PN |

→ **Kết quả:** 9 mẫu gốc + 9 mẫu augmented = **18 mẫu hiệu dụng** cho training.

In [8]:
# Từ điển đồng nghĩa chuyên ngành BĐS
BDS_SYNONYMS = {
    'hẻm xe hơi': 'hẻm ô tô',
    'hẻm ô tô':   'hẻm xe hơi',
    'tầng':        'lầu',
    'lầu':         'tầng',
    'WC':          'nhà vệ sinh',
    'nhà vệ sinh': 'WC',
    'thương lượng': 'TL',
    'TL':           'thương lượng',
    'phòng ngủ':   'PN',
    'PN':           'phòng ngủ',
    'kiên cố':     'vững chắc',
    'đúc BTCT':    'bê tông cốt thép',
}

def augment_one(item):
    """Tạo 1 bản augmented từ 1 mẫu gốc bằng cách thay 1 từ đồng nghĩa."""
    new = item.copy()
    new_target = item['target_text']
    # Chọn ngẫu nhiên 1 từ trong từ điển để thay thế
    key = random.choice(list(BDS_SYNONYMS.keys()))
    if key in new_target:
        new_target = new_target.replace(key, BDS_SYNONYMS[key], 1)
    new['target_text'] = new_target
    return new

if AUGMENT:
    augmented_samples = [augment_one(s) for s in raw_train]
    combined_train    = raw_train + augmented_samples  # Gộp gốc + augmented
    train_ds = to_hf_dataset(combined_train)           # Tokenize lại
    print(f' Data Augmentation hoàn tất!')
    print(f'   Gốc: {len(raw_train)} mẫu → Sau augment: {len(train_ds)} mẫu')
else:
    print('ℹ Bỏ qua augmentation (AUGMENT=False)')
    print(f'   Train dataset: {len(train_ds)} mẫu')

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

 Data Augmentation hoàn tất!
   Gốc: 9 mẫu → Sau augment: 18 mẫu


##  Bước 9: Định Nghĩa Hàm Đánh Giá ROUGE-L

**ROUGE-L** (Recall-Oriented Understudy for Gisting Evaluation — Longest Common Subsequence)
đo độ trùng khớp giữa văn bản AI sinh ra và Ground Truth.

**Ví dụ:**
- Ground Truth: `"Nhà 5 tầng kiên cố, hẻm xe hơi"`
- AI output:    `"Nhà 5 tầng BTCT, hẻm xe hơi rộng"`
- ROUGE-L ≈ 0.75 (75% trùng khớp theo chuỗi dài nhất chung)

Hàm này được gọi **tự động sau mỗi epoch** bởi Seq2SeqTrainer.

In [9]:
# Load ROUGE metric từ HuggingFace evaluate
rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    """
    Tính ROUGE-L sau mỗi epoch Validation.
    Được gọi tự động bởi Seq2SeqTrainer.
    """
    predictions, labels = eval_pred

    # Chuyển token IDs → văn bản
    decoded_preds = tokenizer.batch_decode(
        predictions, skip_special_tokens=True  # Bỏ qua các token đặc biệt [PAD], [EOS]...
    )

    # Labels: thay -100 (padding token đặc biệt) bằng pad_token_id để decode
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Tính ROUGE (trả về dict: rouge1, rouge2, rougeL, rougeLsum)
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=False,  # Không rút gọn từ (tiếng Việt không cần stemmer)
    )

    # Nhân 100 để biểu diễn %
    return {k: round(v * 100, 2) for k, v in result.items()}

print(' Hàm tính ROUGE-L đã sẵn sàng!')

 Hàm tính ROUGE-L đã sẵn sàng!


##  Bước 10: Cấu Hình Seq2SeqTrainer

Cấu hình toàn bộ quy trình training: learning rate schedule, evaluation, early stopping...

**Các chiến lược quan trọng:**
- `eval_strategy='epoch'`: Đánh giá val_loss sau **mỗi epoch** (không phải mỗi step)
- `load_best_model_at_end=True`: Sau training, tự load lại checkpoint tốt nhất
- `fp16=True`: Mixed Precision Training → training nhanh hơn **~2x** trên GPU T4
- `lr_scheduler_type='cosine'`: Giảm learning rate theo đường cong cosine (mượt mà)
- `DataCollatorForSeq2Seq`: Padding động theo batch → tiết kiệm VRAM

In [10]:
# ── Data Collator: Padding động ──────────────────────────────
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    pad_to_multiple_of=8,
    return_tensors='pt',
)

# ── Training Arguments ───────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training loop
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type='cosine',

    # Evaluation & Checkpoint
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    save_safetensors=False,             # FIX: Tắt safetensors để tránh lỗi non-contiguous tensor

    # Generation
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    generation_num_beams=BEAM_SIZE,

    # Logging
    logging_strategy='epoch',
    report_to='none',

    # Tối ưu tốc độ
    fp16=True if device == 'cuda' else False,
    dataloader_num_workers=2,
    seed=SEED,
)

# ── Khởi tạo Trainer ─────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)



##  Bước 11: Fine-Tune ViT5 (Quan Trọng Nhất!)

Bắt đầu quá trình fine-tune. Đây là bước **tốn thời gian nhất** (~15-30 phút).

**Theo dõi trong quá trình training:**
- `train_loss` giảm dần → model đang học
- `eval_loss` giảm dần → model đang tổng quát hóa tốt
- Nếu `eval_loss` tăng trong khi `train_loss` giảm → **Overfitting** → EarlyStopping sẽ dừng



In [12]:
print('🚀 BẮT ĐẦU FINE-TUNE VIT5 (Đã fix lỗi save)...')
print('=' * 55)
print(f'   Model:   {MODEL_NAME}')
print(f'   Train:   {len(train_ds)} mẫu | Val: {len(val_ds)} mẫu')
print(f'   LR={LEARNING_RATE} | Batch={BATCH_SIZE}x{GRAD_ACCUM} | MaxEpoch={EPOCHS}')
print('=' * 55)
print('Theo dõi: train_loss và eval_loss giảm dần là tốt...\n')

# Bắt đầu training — EarlyStopping tự dừng khi cần
t_start     = time.time()
train_result = trainer.train()
elapsed      = time.time() - t_start

# Lưu model tốt nhất (checkpoint có val_loss thấp nhất)
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print('\n' + '=' * 55)
print('✅ FINE-TUNE HOÀN TẤT!')
print(f'   Thời gian:          {elapsed/60:.1f} phút')
print(f'   Train loss cuối:    {train_result.training_loss:.4f}')
print(f'   Tổng steps:         {train_result.global_step}')
print(f'   Model lưu tại:      {FINAL_DIR}/')

🚀 BẮT ĐẦU FINE-TUNE VIT5 (Đã fix lỗi save)...
   Model:   VietAI/vit5-base
   Train:   18 mẫu | Val: 2 mẫu
   LR=0.0001 | Batch=2x4 | MaxEpoch=80
Theo dõi: train_loss và eval_loss giảm dần là tốt...



Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
0,0.054800,2.092349,80.720000,58.060000,57.830000,57.830000
1,0.041100,2.079028,80.280000,58.080000,56.660000,56.660000
2,0.037600,2.088490,78.550000,56.980000,59.260000,59.260000
4,0.032900,2.270146,64.700000,37.340000,46.700000,46.700000
5,0.039100,2.298336,64.430000,34.740000,46.210000,46.210000
6,0.036000,2.352209,72.860000,44.890000,47.320000,47.320000
8,0.026600,2.402352,56.740000,25.410000,35.440000,35.440000
9,0.023900,2.368536,57.780000,22.240000,35.640000,35.640000
10,0.025400,2.363836,60.980000,25.940000,40.490000,40.490000
12,0.025000,2.299500,77.890000,52.530000,57.590000,57.590000



✅ FINE-TUNE HOÀN TẤT!
   Thời gian:          11.9 phút
   Train loss cuối:    0.0265
   Tổng steps:         38
   Model lưu tại:      ./vit5_best/


##  Bước 12: Vẽ Loss Curve

Vẽ biểu đồ **Train Loss vs Validation Loss** qua các epoch.

**Đọc biểu đồ:**
- Cả 2 đường đều giảm →  Model đang học tốt
- `train_loss` giảm nhưng `val_loss` tăng →  Overfitting
- Đường xanh lá dọc = Epoch tốt nhất (val_loss thấp nhất) → EarlyStopping dừng tại đây

>  **Lưu ảnh `loss_curve.png`** để đính kèm vào báo cáo!

In [14]:
# Lấy lịch sử training từ trainer.state
history = trainer.state.log_history

# Tách riêng train loss và val loss
train_loss_hist = [(h['epoch'], h['loss'])      for h in history if 'loss' in h and 'eval_loss' not in h]
val_loss_hist   = [(h['epoch'], h['eval_loss']) for h in history if 'eval_loss' in h]
rouge_hist      = [(h['epoch'], h.get('eval_rougeL', 0)) for h in history if 'eval_rougeL' in h]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('ViT5 Fine-tuning — Loss Curve | Đinh Thị Mai Anh', fontsize=13, fontweight='bold')

# ── Biểu đồ 1: Train Loss vs Val Loss ────────────────────────
if train_loss_hist:
    te, tv = zip(*train_loss_hist)
    axes[0].plot(te, tv, 'b-o', label='Train Loss', linewidth=2, markersize=4)
if val_loss_hist:
    ve, vv = zip(*val_loss_hist)
    axes[0].plot(ve, vv, 'r-s', label='Val Loss', linewidth=2, markersize=4)
    # Đánh dấu epoch tốt nhất
    best_epoch = ve[list(vv).index(min(vv))]
    axes[0].axvline(x=best_epoch, color='green', linestyle='--', alpha=0.8,
                    label=f'Best epoch = {best_epoch:.0f}')
    print(f'📌 Val Loss tốt nhất: {min(vv):.4f} tại epoch {best_epoch:.0f}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Train vs Validation Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# ── Biểu đồ 2: ROUGE-L theo epoch ───────────────────────────
if rouge_hist:
    re_, rv = zip(*rouge_hist)
    axes[1].plot(re_, rv, 'g-^', label='ROUGE-L (Val)', linewidth=2, markersize=4)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('ROUGE-L (%)')
    axes[1].set_title('ROUGE-L trên tập Val')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    print(f'📌 ROUGE-L tốt nhất trên Val: {max(rv):.2f}%')
else:
    axes[1].text(0.5, 0.5, 'ROUGE-L\n(hiển thị sau training)',
                 ha='center', va='center', transform=axes[1].transAxes, fontsize=12)

plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📸 Đã lưu: loss_curve.png — Dùng ảnh này đính kèm vào báo cáo!')

📌 Val Loss tốt nhất: 2.0790 tại epoch 2
📌 ROUGE-L tốt nhất trên Val: 62.74%

📸 Đã lưu: loss_curve.png — Dùng ảnh này đính kèm vào báo cáo!


##  Bước 13: Inference Trên 9 Căn Test (Beam Search)

Chạy mô hình đã fine-tune trên **9 căn Test** để sinh ra kết quả chuẩn hóa.

**Beam Search là gì?**
- Thay vì chọn từ có xác suất cao nhất (Greedy), Beam Search thử **4 hướng song song**
- Sau đó chọn chuỗi từ có tổng xác suất cao nhất → câu mượt mà, đầy đủ hơn

**Đồng thời đo Latency:** Thời gian inference mỗi căn (giây) → điền vào `performance.json`.

In [15]:
# ── Load lại model tốt nhất từ checkpoint đã lưu ──────────────
print(f'📂 Load model tốt nhất từ {FINAL_DIR}...')
best_model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_DIR).to(device)
best_tok   = AutoTokenizer.from_pretrained(FINAL_DIR)
best_model.eval()

gen_config = GenerationConfig(
    max_new_tokens=MAX_TARGET_LEN,
    num_beams=BEAM_SIZE,
    length_penalty=LENGTH_PENALTY,
    no_repeat_ngram_size=NO_REPEAT_NGRAM,
    early_stopping=True,
)


# ── Hàm inference ──────────────────────────────────────────────
def infer_one(raw_text):
    """Inference 1 căn nhà → trả về (text sinh ra, latency giây)."""
    inp = best_tok(
        TASK_PREFIX + raw_text,
        return_tensors='pt',
        max_length=MAX_INPUT_LEN,
        truncation=True,
    ).to(device)
    t0 = time.time()
    with torch.no_grad():
        output_ids = best_model.generate(**inp, generation_config=gen_config)
    latency = time.time() - t0
    decoded = best_tok.decode(output_ids[0], skip_special_tokens=True)
    return decoded, latency


# ── Hàm trích xuất specs từ raw text (fallback khi JSON không có) ──
def extract_specs_from_text(mo_ta_tho):
    """
    Trích xuất 9 specs kỹ thuật từ dòng đầu của mô tả thô.
    Dùng khi file JSON (test.json) không có sẵn key 'extracted_specs'.

    Dòng đầu có cấu trúc:
    <Tên đường> <DT> <Số tầng> <Ngang> <Dài> <Giá> tỷ <Khu vực> <Quận> ...
    Ví dụ: 'Cao Thắng 35 5 3.4 11 8.9 tỷ Bàn Cờ Quận 3 5-10 tỷ ...'
    """
    specs = {
        'duong': '', 'phuong': '', 'quan': '',
        'gia_ty': 0.0, 'dien_tich_m2': 0.0, 'so_tang': 0,
        'mat_tien_m': 0.0, 'rong_hem_m': 0.0, 'phan_loai_hem': '',
    }
    if not mo_ta_tho:
        return specs

    line1 = mo_ta_tho.strip().split('\n')[0]

    # Tên đường (phần chữ trước dãy số đầu tiên)
    m = re.match(r'^([A-Za-zÀ-ỹ\s\.]+?)\s+[\d]', line1)
    if m:
        duong_raw = re.sub(r'\s+\d+$', '', m.group(1)).strip()
        specs['duong'] = duong_raw

    # Diện tích (số đầu tiên trong chuỗi số trước 'tỷ')
    m = re.search(r'(?:^|\s)([\d]+(?:\.[\d]+)?)(?:/[\d\.]+)?\s+\d+\s+[\d\.]+\s+[\d\.]+\s+[\d\.]+\s+tỷ', line1)
    if m:
        try: specs['dien_tich_m2'] = float(m.group(1))
        except: pass

    # Số tầng
    m = re.search(r'[\d\.]+(?:/[\d\.]+)?\s+(\d+)\s+[\d\.]+\s+[\d\.]+\s+[\d\.]+\s+tỷ', line1)
    if m:
        try: specs['so_tang'] = int(m.group(1))
        except: pass

    # Mặt tiền (m)
    m = re.search(r'[\d\.]+(?:/[\d\.]+)?\s+\d+\s+([\d\.]+)\s+[\d\.]+\s+[\d\.]+\s+tỷ', line1)
    if m:
        try: specs['mat_tien_m'] = float(m.group(1))
        except: pass

    # Giá (số trước 'tỷ')
    m = re.search(r'([\d]+(?:[,.][\d]+)?)\s*tỷ', line1)
    if m:
        try: specs['gia_ty'] = float(m.group(1).replace(',', '.'))
        except: pass

    # Quận / Phường
    m = re.search(r'Phường\s+([\w\s]+?)\s+Quận\s+(\w+)', mo_ta_tho)
    if m:
        specs['phuong'] = 'Phường ' + m.group(1).strip()
        specs['quan']   = 'Quận '   + m.group(2).strip()
    else:
        m = re.search(r'Quận\s+(\w+)', mo_ta_tho)
        if m: specs['quan'] = 'Quận ' + m.group(1).strip()

    # Phân loại hẻm
    full = mo_ta_tho.lower()
    if re.search(r'hẻm.*?xe tải|xe tải.*?hẻm', full):
        specs['phan_loai_hem'] = 'Hẻm xe tải'
    elif re.search(r'hẻm.*?ô tô|ô tô.*?hẻm|hẻm.*?xe hơi|xe hơi.*?hẻm', full):
        specs['phan_loai_hem'] = 'Hẻm xe hơi'
    elif 'hẻm' in full:
        specs['phan_loai_hem'] = 'Hẻm'

    # Rộng hẻm
    m = re.search(r'hẻm\s+(?:rộng\s+)?([\d\.]+)\s*m', full)
    if m:
        try: specs['rong_hem_m'] = float(m.group(1))
        except: pass

    return specs


# ── Hàm tạo tiêu đề chuẩn từ specs ────────────────────────────
def build_title_from_specs(specs):
    """
    Tạo tiêu đề BĐS từ specs:
    'Đường Quận X - DT m2 (Ngang m) - X Tầng - Loại hẻm - Giá Tỷ'

    Dùng specs thay vì text model sinh ra vì:
    - ViT5 với 9 mẫu train không đủ để học format tiêu đề chuẩn.
    - Specs đảm bảo thông số chính xác, không bị hallucinate.
    """
    duong    = specs.get('duong',         '').strip()
    quan     = specs.get('quan',          '').strip()
    gia      = specs.get('gia_ty',         0)
    dt       = specs.get('dien_tich_m2',   0)
    tang     = specs.get('so_tang',        0)
    ngang    = specs.get('mat_tien_m',     0)
    hem_type = specs.get('phan_loai_hem', '').strip()

    parts = []
    if duong and quan:
        parts.append(f'{duong} {quan}')
    elif duong:
        parts.append(duong)
    if dt and ngang:
        parts.append(f'{dt}m2 ({ngang}m ngang)')
    elif dt:
        parts.append(f'{dt}m2')
    if tang:
        parts.append(f'{tang} Tầng')
    if hem_type:
        parts.append(hem_type)
    if gia:
        parts.append(f'{gia} Tỷ')

    return ' - '.join(parts) if parts else ''


# ── Hàm làm sạch raw output từ model ───────────────────────────
def clean_model_output(raw_output):
    """
    Làm sạch text model sinh ra để dùng làm predicted_description.
    Vấn đề: model hay sinh fragment rác ở đầu ('*9 TỶ', '*ÁT XE HƠI'...)
    do BOS token không ổn định với dataset chỉ 9 mẫu train.
    """
    text = raw_output.strip()

    # Cắt bỏ fragment rác nếu bắt đầu bằng '*'
    if text.startswith('*'):
        REAL_STARTS = [
            r'Vị trí:', r'Hẻm:', r'Kết cấu:', r'Diện tích:',
            r'TRUNG TÂM', r'SIÊU PHẨM', r'KHU VỰC', r'NHÀ ', r'– Vị',
        ]
        pattern = '(' + '|'.join(REAL_STARTS) + ')'
        m = re.search(pattern, text)
        if m:
            text = text[m.start():]

    # Tách đoạn theo double-space → ghép lại bằng \n\n
    paragraphs = [p.strip() for p in re.split(r'  +', text) if p.strip()]
    return '\n\n'.join(paragraphs)


# ── Chạy Inference trên toàn bộ raw_test ───────────────────────
print(f'\n🔮 Chạy Beam Search (beam={BEAM_SIZE}) trên {len(raw_test)} căn Test...')
print(f'   (test.json có {len(raw_test)} entries — sẽ inference tất cả)')
print('=' * 60)

predictions = []
latencies   = []

for i, sample in enumerate(raw_test):
    # Bước 1: Inference → raw text từ model
    raw_output, lat = infer_one(sample['input_text'])
    latencies.append(lat)

    # Bước 2: Lấy specs
    # Ưu tiên: specs có sẵn trong JSON → fallback: trích xuất từ raw text
    raw_item = sample.get('raw_item', {})
    specs    = raw_item.get('extracted_specs')  # None nếu không có key
    if not specs:
        # test.json không có extracted_specs → tự trích xuất
        specs = extract_specs_from_text(sample['input_text'])

    # Bước 3: Tạo title từ specs (thông số đúng, không hallucinate)
    title = build_title_from_specs(specs)

    # Bước 4: Làm sạch raw output → dùng làm description
    desc = clean_model_output(raw_output)

    record = {
        'id':                    sample['id'],
        'raw_input_cleaned':     sample['input_text'],
        'predicted_title':       title,
        'predicted_description': desc,
        'extracted_specs':       specs,
    }
    predictions.append(record)

    pid = sample['id']
    print(f'\n📌 Căn #{i+1} | ID={pid} | {lat:.2f}s')
    print(f'   [TITLE] {title}')
    print(f'   [DESC ] {desc[:100]}...')

avg_lat = np.mean(latencies)
print('\n' + '=' * 60)
print(f'✅ Inference hoàn tất!')
print(f'   Tổng: {len(predictions)} predictions')
print(f'   Latency trung bình: {avg_lat:.3f}s/căn')
print(f'   Min/Max: {min(latencies):.3f}s / {max(latencies):.3f}s')
has_title = all(bool(p["predicted_title"]) for p in predictions)
has_desc  = all(bool(p["predicted_description"]) for p in predictions)
print(f'   Tất cả có predicted_title: {has_title}')
print(f'   Tất cả có predicted_description: {has_desc}')


📂 Load model tốt nhất từ ./vit5_best...

🔮 Chạy Beam Search (beam=4) trên 26 căn Test...
   (test.json có 26 entries — sẽ inference tất cả)

📌 Căn #1 | ID=SYS-MP75Z7G0-H3 | 1.99s
   [TITLE] Cao Thắng Quận 10 - 44.7m2 (4.5m ngang) - 5 Tầng - Hẻm xe hơi - 11.9 Tỷ
   [DESC ] Cách Mạng Tháng Hai Quận 10 chỉ vài phút di chuyển.

Kết cấu: Nhà 4 tầng, khung BTCT kiên cố, thiết ...

📌 Căn #2 | ID=SYS-MP75ZR8R-C4 | 4.57s
   [TITLE] Cao Thắng Quận 10 - 39.0m2 (5.3m ngang) - 4 Tầng - Hẻm xe hơi - 14.58 Tỷ
   [DESC ] Mặt tiền Cao Thắng Quận 10 - 39m2 (4 tầng) - Hẻm xe hơi thông - Ngang lớn - 14.8 Tỷ

Ở ngay hoặc Kin...

📌 Căn #3 | ID=SYS-MP760LYQ-AC | 3.26s
   [TITLE] Cao Thắng Quận 10 - 60.0m2 (5.0m ngang) - 3 Tầng - Hẻm xe hơi - 15.5 Tỷ
   [DESC ] Cách Mạng Tháng Tám Quận 10 - 60m2 (13.5x15.5) - Hẻm xe hơi vào nhà - 65 triệu/tháng (25.000đ)

Hẻm:...

📌 Căn #4 | ID=SYS-MP760YOH-BW | 3.28s
   [TITLE] Đồng Nai Quận 10 - 71.2m2 (4.3m ngang) - 5 Tầng - Hẻm - 22.8 Tỷ
   [DESC ] Cách Mạng Tháng Tám Quậ

##  Bước 14: Tự Tính ROUGE-L Trên Val (2 Mẫu Có GT)

Tính ROUGE-L trên **2 mẫu Val** có Ground Truth để điền vào báo cáo.



In [16]:
print('📊 Tính ROUGE-L trên tập Val (2 mẫu có Ground Truth)...')

val_preds, val_refs = [], []
for s in raw_val:
    gen, _ = infer_one(s['input_text'])
    val_preds.append(gen)
    val_refs.append(s['target_text'])

    print(f'\n📌 ID: {s["id"]}')
    print(f'   GT:  {s["target_text"][:100]}...')
    print(f'   Pred:{gen[:100]}...')

rouge_result = rouge.compute(predictions=val_preds, references=val_refs)

print('\n📊 KẾT QUẢ ROUGE TRÊN VAL:')
print(f'   ROUGE-1:  {rouge_result["rouge1"]*100:.2f}%')
print(f'   ROUGE-2:  {rouge_result["rouge2"]*100:.2f}%')
print(f'   ROUGE-L:  {rouge_result["rougeL"]*100:.2f}%  ← CHỈ SỐ CHÍNH')

rouge_val_score = rouge_result['rougeL'] * 100
print(f'\n📝 Điền vào báo cáo: ROUGE-L (Val) = {rouge_val_score:.2f}%')

📊 Tính ROUGE-L trên tập Val (2 mẫu có Ground Truth)...

📌 ID: SYS-MP75438B-O8
   GT:  Hẻm xe hơi Lê Văn Sỹ Quận 3 - 46m2 (4.6x10) - Ngang lớn - 12.8 Tỷ | Ở ngay

TRUNG TÂM QUẬN 3 ĐƯỜNG T...
   Pred:Cách Mạng Tháng Tám Quận 3 - 46m2 (4.6x10) - Hẻm xe hơi đỗ cửa - Ngang lớn - 12.8 Tỷ  Ở ngay hoặc Ki...

📌 ID: SYS-MP75GT4Y-GN
   GT:  Huỳnh Văn Bánh Phú Nhuận - 30m2 (3.7x8) - Hẻm xe hơi VIP - 8.9 Tỷ | Full nội thất xịn

KHU VIP PHÚ N...
   Pred:Cách Mạng Tháng Tám Phú Nhuận - 30m2 (3.7x8) - Hẻm xe hơi thông - Ngang lớn - 8.9 Tỷ  Ở ngay TRUNG T...

📊 KẾT QUẢ ROUGE TRÊN VAL:
   ROUGE-1:  70.40%
   ROUGE-2:  47.97%
   ROUGE-L:  46.64%  ← CHỈ SỐ CHÍNH

📝 Điền vào báo cáo: ROUGE-L (Val) = 46.64%


## Bước 14.5: Tính BERTScore trên Val

**BERTScore** đo độ tương đồng ngữ nghĩa giữa văn bản sinh ra và Ground Truth
bằng cách dùng embedding từ mô hình BERT — khác với ROUGE chỉ đếm n-gram khớp.

| Metric | Nguyên lý | Ưu điểm |
|---|---|---|
| ROUGE-L | Đếm chuỗi con chung dài nhất (LCS) | Nhanh, không cần GPU |
| **BERTScore** | So sánh vector ngữ nghĩa (cosine sim) | Hiểu nghĩa, tốt hơn với tiếng Việt |

> **Lưu ý:** Dùng `lang='vi'` để BERTScore chọn đúng model embedding cho tiếng Việt
> (mặc định dùng `bert-base-multilingual-cased`).


In [21]:
# Cài đặt bert-score (chỉ cần 1 lần)
!pip install bert-score -q

from bert_score import score as bert_score_fn

print('\U0001f4ca Tính BERTScore trên Val (2 mẫu có Ground Truth)...')
print('=' * 60)

# val_preds và val_refs đã được tạo ở Cell ROUGE-L phía trên
# val_preds: list các câu model sinh ra
# val_refs:  list các câu Ground Truth

# Tính BERTScore — lang='vi' để dùng multilingual BERT
P, R, F1 = bert_score_fn(
    cands=val_preds,
    refs=val_refs,
    lang='vi',
    verbose=True,
)

# Lấy F1 trung bình (chỉ số chính)
bertscore_f1        = round(F1.mean().item() * 100, 2)
bertscore_precision = round(P.mean().item() * 100, 2)
bertscore_recall    = round(R.mean().item() * 100, 2)

print('\n\U0001f4ca KẾT QUẢ BERTScore TRÊN VAL (2 mẫu):')
print(f'   Precision: {bertscore_precision:.2f}%')
print(f'   Recall:    {bertscore_recall:.2f}%')
print(f'   F1:        {bertscore_f1:.2f}%  \u2190 CHỈ SỐ CHÍNH')
print()
print('So sánh với ROUGE-L:')
print(f'   ROUGE-L:   {rouge_val_score:.2f}%')
print(f'   BERTScore: {bertscore_f1:.2f}%')
diff = bertscore_f1 - rouge_val_score
if diff > 0:
    print(f'   → BERTScore cao hơn ROUGE-L {diff:.2f}pp (ngữ nghĩa tốt hơn n-gram)')
else:
    print(f'   → BERTScore thấp hơn ROUGE-L {abs(diff):.2f}pp')
print(f'\n\U0001f4dd Điền vào báo cáo: BERTScore F1 (Val) = {bertscore_f1:.2f}%')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00
📊 Tính BERTScore trên Val (2 mẫu có Ground Truth)...


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.34 seconds, 5.94 sentences/sec

📊 KẾT QUẢ BERTScore TRÊN VAL (2 mẫu):
   Precision: 82.31%
   Recall:    76.55%
   F1:        79.32%  ← CHỈ SỐ CHÍNH

So sánh với ROUGE-L:
   ROUGE-L:   46.64%
   BERTScore: 79.32%
   → BERTScore cao hơn ROUGE-L 32.68pp (ngữ nghĩa tốt hơn n-gram)

📝 Điền vào báo cáo: BERTScore F1 (Val) = 79.32%


##  Bước 15: Thực Nghiệm So Sánh Beam Size 1→5

Chạy thử các Beam Size khác nhau trên 1 mẫu Val để tìm beam size **cho ROUGE-L tốt nhất**.

| Beam Size | Đặc điểm |
|---|---|
| 1 | Greedy Search — nhanh nhất, kém nhất |
| 2-3 | Cân bằng tốt giữa tốc độ và chất lượng |
| 4-5 | Chất lượng cao hơn, chậm hơn một chút |

> Sau khi xem kết quả, cập nhật `BEAM_SIZE` ở Cell 3 nếu muốn dùng beam khác.

In [17]:
print('🔬 So sánh Beam Size 1 → 5 trên 1 mẫu Val...')

s_val = raw_val[0]
beam_results = []
for b in [1, 2, 3, 4, 5]:
    gc = GenerationConfig(max_new_tokens=MAX_TARGET_LEN, num_beams=b, length_penalty=1.2, no_repeat_ngram_size=3, early_stopping=True)
    inp = best_tok(TASK_PREFIX + s_val['input_text'], return_tensors='pt', max_length=MAX_INPUT_LEN, truncation=True).to(device)
    t = time.time()
    with torch.no_grad():
        out = best_model.generate(**inp, generation_config=gc)
    lat = time.time() - t
    decoded = best_tok.decode(out[0], skip_special_tokens=True)
    r = rouge.compute(predictions=[decoded], references=[s_val['target_text']])
    rl = r['rougeL'] * 100
    beam_results.append({'beam': b, 'rouge_l': rl, 'latency': lat})
    print(f'Beam={b}: ROUGE-L={rl:.2f}% | {lat:.3f}s')

best_b = max(beam_results, key=lambda x: x['rouge_l'])
print(f'\n🏆 Beam size tốt nhất: {best_b["beam"]} (ROUGE-L={best_b["rouge_l"]:.2f}%)')

🔬 So sánh Beam Size 1 → 5 trên 1 mẫu Val...
Beam=1: ROUGE-L=48.02% | 2.744s
Beam=2: ROUGE-L=52.74% | 4.011s
Beam=3: ROUGE-L=47.53% | 4.126s
Beam=4: ROUGE-L=48.99% | 3.521s
Beam=5: ROUGE-L=49.35% | 4.196s

🏆 Beam size tốt nhất: 2 (ROUGE-L=52.74%)


##  Bước 16: Lưu File `predictions.json`

Lưu kết quả inference ra file JSON theo **đúng định dạng DoD** của nhóm.

**Cấu trúc 1 entry:**
```json
{
  "id": "SYS-MP764LWN-9X",
  "raw_input_cleaned": "...",
  "predicted_title": "...",
  "predicted_description": "...",
  "extracted_specs": { "duong": "...", "gia_ty": 8.9, ... }
}
```



In [18]:
# Tên file theo đúng quy ước DoD: MaiAnh_ViT5_predictions.json
PRED_FILE = 'MaiAnh_ViT5_predictions.json'

# Lưu với ensure_ascii=False để giữ nguyên tiếng Việt
with open(PRED_FILE, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f'Saved: {PRED_FILE}')
print(f'  So entries:             {len(predictions)}')
print(f'  Co predicted_title:     {all(bool(p["predicted_title"]) for p in predictions)}')
print(f'  Co predicted_description: {all(bool(p["predicted_description"]) for p in predictions)}')

# Preview entry dau tien de kiem tra
print('\nPreview entry[0]:')
print(json.dumps(predictions[0], ensure_ascii=False, indent=2)[:400] + '...')

# Download ve may tinh
files.download(PRED_FILE)
print(f'\nDownloaded: {PRED_FILE}')


Saved: MaiAnh_ViT5_predictions.json
  So entries:             26
  Co predicted_title:     True
  Co predicted_description: True

Preview entry[0]:
{
  "id": "SYS-MP75Z7G0-H3",
  "raw_input_cleaned": "Cao Thắng 44.7/50 5 4.5 11 11.9 tỷ Hòa Hưng Quận 10 10-15 tỷ HĐ Nguyễn Duy Bảo Thiên Địa  H3GB nguồn Đầu chủ Nguyễn Duy Bảo - Thiên Địa\n✨ 4/5/2026 Chủ đã đồng ý giảm 600tr 👉 Giá mới 11.9 Tỷ. Chốt 🤝\n\n👉 Chú ý: ACE đi khảo sát vui lòng chụp 1 ảnh selfie 🤳 gửi ĐC để nhận sổ nhé. Chốt 🤝\n\nNHÀ ĐẸP TRUNG TÂM QUẬN 10 – HẺM XE HƠI CAO THẮNG – KHU DÂN...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Downloaded: MaiAnh_ViT5_predictions.json


##  Bước 17: Lưu File `performance.json`

Ghi lại toàn bộ thông số hiệu năng: latency, hyperparameters, thông tin GPU.

File này dùng để so sánh tốc độ giữa 5 mô hình.

In [24]:
# Tên file theo đúng quy ước DoD: MaiAnh_ViT5_performance.json
PERF_FILE = 'MaiAnh_ViT5_performance.json'

# Lấy thông tin training (nếu đã train trong session này)
try:
    final_loss  = round(train_result.training_loss, 4)
    total_steps = train_result.global_step
except NameError:
    final_loss  = None
    total_steps = None

# Lấy BERTScore nếu đã chạy (Cell BERTScore phía trên)
try:
    _bertscore_f1        = bertscore_f1
    _bertscore_precision = bertscore_precision
    _bertscore_recall    = bertscore_recall
except NameError:
    _bertscore_f1 = _bertscore_precision = _bertscore_recall = None

perf_data = {
    # Thông tin cơ bản
    'model_name':                MODEL_NAME,
    'average_latency_seconds':   round(avg_lat, 4),
    'estimated_cost_vnd_per_1k': 0,

    # Latency chi tiết
    'latency_per_sample': [round(l, 4) for l in latencies],

    # Kết quả đánh giá
    'rouge_l_on_val_2_samples': round(rouge_val_score, 2),
    'bertscore_on_val_2_samples': {
        'f1':        _bertscore_f1,
        'precision': _bertscore_precision,
        'recall':    _bertscore_recall,
        'lang':      'vi',
    },
    'note': (
        f'Val: {len(raw_val)} mau co GT. '
        f'Test: {len(raw_test)} mau khong co GT — danh gia dinh tinh.'
    ),

    # Thông tin dataset
    'dataset_info': {
        'train_samples':          len(raw_train),
        'val_samples':            len(raw_val),
        'test_samples':           len(raw_test),
        'test_has_extracted_specs': any(
            'extracted_specs' in s.get('raw_item', {})
            for s in raw_test
        ),
    },

    # Thông tin training
    'training_info': {
        'final_train_loss': final_loss,
        'total_steps':      total_steps,
    },

    # Hyperparameters
    'hyperparameters': {
        'learning_rate':     LEARNING_RATE,
        'batch_size':        BATCH_SIZE,
        'grad_accumulation': GRAD_ACCUM,
        'effective_batch':   BATCH_SIZE * GRAD_ACCUM,
        'max_input_len':     MAX_INPUT_LEN,
        'max_target_len':    MAX_TARGET_LEN,
        'beam_size':         BEAM_SIZE,
        'length_penalty':    LENGTH_PENALTY,
        'no_repeat_ngram':   NO_REPEAT_NGRAM,
        'weight_decay':      WEIGHT_DECAY,
        'data_augmentation': AUGMENT,
        'seed':              SEED,
    },

    # Phần cứng
    'hardware': {
        'device':      device,
        'gpu_name':    torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'gpu_vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
                       if torch.cuda.is_available() else 0,
    },
}

with open(PERF_FILE, 'w', encoding='utf-8') as f:
    json.dump(perf_data, f, ensure_ascii=False, indent=2)

print(f'Saved: {PERF_FILE}')
print(json.dumps(perf_data, ensure_ascii=False, indent=2))

files.download(PERF_FILE)
print(f'\nDownloaded: {PERF_FILE}')


Saved: MaiAnh_ViT5_performance.json
{
  "model_name": "VietAI/vit5-base",
  "average_latency_seconds": 4.1961,
  "estimated_cost_vnd_per_1k": 0,
  "latency_per_sample": [
    1.9899,
    4.5668,
    3.2615,
    3.2761,
    4.991,
    4.5441,
    5.6495,
    4.3229,
    3.9084,
    2.8726,
    4.7167,
    4.3419,
    4.1946,
    2.5721,
    5.5704,
    4.0669,
    3.8247,
    4.6302,
    5.6993,
    3.795,
    5.5123,
    4.5433,
    4.0813,
    4.4322,
    3.0335,
    4.7009
  ],
  "rouge_l_on_val_2_samples": 46.64,
  "bertscore_on_val_2_samples": {
    "f1": 79.32,
    "precision": 82.31,
    "recall": 76.55,
    "lang": "vi"
  },
  "note": "Val: 2 mau co GT. Test: 26 mau khong co GT — danh gia dinh tinh.",
  "dataset_info": {
    "train_samples": 9,
    "val_samples": 2,
    "test_samples": 26,
    "test_has_extracted_specs": false
  },
  "training_info": {
    "final_train_loss": 0.0265,
    "total_steps": 38
  },
  "hyperparameters": {
    "learning_rate": 0.0001,
    "batch_size":

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Downloaded: MaiAnh_ViT5_performance.json


## 📥 Bước 18: Download Tất Cả File


In [20]:
print('Download tat ca file ban giao...')
print('=' * 50)

# Danh sach file can download — ten dung theo quy uoc DoD
deliverables = [
    ('MaiAnh_ViT5_predictions.json', f'Ket qua {len(raw_test)} can Test'),
    ('MaiAnh_ViT5_performance.json', 'Thong so hieu nang + hyperparams'),
    ('loss_curve.png',               'Bieu do Loss Curve (Train vs Val)'),
]

for fname, desc in deliverables:
    if os.path.exists(fname):
        files.download(fname)
        print(f'  OK {fname:45s} --- {desc}')
    else:
        print(f'  MISSING {fname:45s} --- Chay lai cell truoc!')

print('\nNho download them thu cong:')
print('  -> File .ipynb: File -> Download -> Download .ipynb')
print('  -> Link Colab:  Share -> Anyone with link -> Viewer')


Download tat ca file ban giao...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  OK MaiAnh_ViT5_predictions.json                  --- Ket qua 26 can Test


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  OK MaiAnh_ViT5_performance.json                  --- Thong so hieu nang + hyperparams


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  OK loss_curve.png                                --- Bieu do Loss Curve (Train vs Val)

Nho download them thu cong:
  -> File .ipynb: File -> Download -> Download .ipynb
  -> Link Colab:  Share -> Anyone with link -> Viewer
